First, create the model. This must match the model used in the interactive training notebook.

In [ ]:
import torch
import torchvision
from scripts.resnet_sensor_fusion import create_resnet18_sensor_fusion

CATEGORIES = ['apex_sensor']

device = torch.device('cuda')
model = create_resnet18_sensor_fusion(2 * len(CATEGORIES), pretrained=False)
model = model.cuda().eval().half()

In [ ]:
model.load_state_dict(torch.load('best_steering_model_xy.pth'))

Convert and optimize the model using ``torch2trt`` for faster inference with TensorRT.  Please see the [torch2trt](https://github.com/NVIDIA-AI-IOT/torch2trt) readme for more details.

> This optimization process can take a couple minutes to complete. 

In [ ]:
from torch2trt import torch2trt

data = torch.zeros((1, 3, 224, 224)).cuda().half()
sensors = torch.zeros((1, 2)).cuda().half()  # ToF pair; dummy zeros, shape only

model_trt = torch2trt(model, [data, sensors], fp16_mode=True)

Save the optimized model using the cell below

In [ ]:
torch.save(model_trt.state_dict(), 'road_following_model_sensor_trt.pth')